In [ ]:
from pathlib import Path
import os,subprocess
import sys
import wandb

repo = Path(os.environ.get("NANOWM_REPO", Path.cwd())).resolve()
os.chdir(repo)
sys.path.insert(0, str(repo / "src"))

wandb.login()

### Set output path

In [ ]:
import os
from pathlib import Path

repo = Path(os.environ.get("NANOWM_REPO", Path.cwd())).resolve()
os.environ["RESULTS_DIR"] = str(Path(os.environ.get("NANOWM_RESULTS_DIR", repo.parent / "outputs/trained-wm")).resolve())



### Define training settings

In [ ]:
overrides = [
    "dataset=dino_wm/pusht",
    "experiment=dino_wm_pusht",
    "model=nanowm_b2",

    f"dataset.loader.data_path_train={os.environ.get('WM_DATA_TRAIN', str(repo.parent / 'outputs/wm_training_data/pushT_small/train'))}",
    f"+dataset.loader.original_data_path_train={os.environ.get('WM_ORIGINAL_DATA_TRAIN', str(repo.parent / 'outputs/wm_training_data/pushT_original/train'))}",
    "+dataset.loader.use_ER=true",
    f"dataset.loader.data_path_val={os.environ.get('WM_DATA_VAL', str(repo.parent / 'outputs/wm_training_data/pushT_small/val'))}",
    f"+dataset.loader.original_data_path_val={os.environ.get('WM_ORIGINAL_DATA_VAL', str(repo.parent / 'outputs/wm_training_data/pushT_original/val'))}",
    "dataset.loader.use_relative_actions=true",
    "dataset.loader.split_ratio=1.0",
    "dataset.loader.train_slice_mode=exhaustive",
    "dataset.loader.val_slice_mode=exhaustive",
    "dataset.loader.action_scale=100.0",
    "dataset.loader.with_velocity=false",

    f"experiment.pretrained={os.environ.get('NANOWM_PRETRAINED_CHECKPOINT', str(repo / 'nanowm-pusht/assets/checkpoint/model.safetensors'))}",
    "experiment.training.max_steps=50",
    "experiment.training.optimizer.lr=1e-5",
    "experiment.training.batch_size=6",
    "experiment.training.gradient_accumulation=1",
    "experiment.infra.mixed_precision=true",  # bf16-mixed
    "experiment.infra.compile=true",  # torch.compile
    "wandb.enabled=false",

]



### Run the training

In [ ]:
import subprocess
import sys

repo = Path(os.environ.get("NANOWM_REPO", Path.cwd())).resolve()

subprocess.run(
    [str(repo / "nanowm"), "python", "src/main.py", *overrides],
    cwd=repo,
    check=True,
)

### Create model.safetensor

In [ ]:
from pathlib import Path
import torch
from safetensors.torch import save_file
import os

results_root = Path(os.environ.get("NANOWM_RESULTS_DIR", repo.parent / "outputs/trained-wm"))
candidates = list(results_root.glob("*-NanoWM-B-2-F4S5-pusht/checkpoints/latest/latest-*.ckpt"))
if not candidates:
    raise FileNotFoundError(f"No PushT fine-tuning checkpoint found under {results_root}")
src = max(candidates, key=lambda path: path.stat().st_mtime)
dst = src.parents[2] / "model.safetensors"

ckpt = torch.load(src, map_location="cpu", weights_only=False)
state = ckpt["state_dict"]

weights = {}
for key, value in state.items():
    if not key.startswith("model."):
        continue
    key = key.removeprefix("model.").removeprefix("_orig_mod.")
    weights[key] = value.detach().cpu().contiguous()

assert weights, "No model weights found in the checkpoint"
save_file(weights, str(dst))
print(f"Saved {len(weights)} tensors to {dst}")